In [1]:
import sys
sys.path.append("..")

import numpy as np
import wfdb
from sklearn.metrics import classification_report
from src.pipeline import load_model, analyze, CLASSES, AAMI_MAP

bundle = load_model("../models/ecg_rf.joblib")
DATA_DIR = "../data/mitdb"

def evaluate(rec):
    record = wfdb.rdrecord(f"{DATA_DIR}/{rec}")
    ann = wfdb.rdann(f"{DATA_DIR}/{rec}", "atr")
    ch = record.sig_name.index("MLII")
    res = analyze(record.p_signal[:, ch], record.fs, bundle)

    keep = [i for i, s in enumerate(ann.symbol)
            if s in AAMI_MAP and AAMI_MAP[s] in CLASSES]
    a_samples = ann.sample[keep]
    a_labels = np.array([CLASSES.index(AAMI_MAP[ann.symbol[i]]) for i in keep])

    y_true, y_pred = [], []
    for p, c in zip(res["peaks"], res["pred"]):
        j = np.argmin(np.abs(a_samples - p))
        if abs(a_samples[j] - p) <= 36:      # حداکثر ۰٫۱ ثانیه فاصله
            y_true.append(a_labels[j])
            y_pred.append(c)

    print(f"===== record {rec}")
    print("ضربان‌های برچسب‌دار:", len(a_samples),
          "| پیدا شده:", len(res["peaks"]),
          "| جور شده:", len(y_true))
    print(classification_report(y_true, y_pred, labels=[0, 1, 2, 3],
                                target_names=CLASSES, digits=3, zero_division=0))

for rec in ["100", "200", "233"]:
    evaluate(rec)

===== record 100
ضربان‌های برچسب‌دار: 2273 | پیدا شده: 2271 | جور شده: 2271
              precision    recall  f1-score   support

           N      1.000     1.000     1.000      2237
           S      1.000     1.000     1.000        33
           V      1.000     1.000     1.000         1
           F      0.000     0.000     0.000         0

    accuracy                          1.000      2271
   macro avg      0.750     0.750     0.750      2271
weighted avg      1.000     1.000     1.000      2271

===== record 200
ضربان‌های برچسب‌دار: 2601 | پیدا شده: 2598 | جور شده: 2595
              precision    recall  f1-score   support

           N      0.987     0.983     0.985      1742
           S      0.266     0.833     0.403        30
           V      1.000     0.929     0.963       822
           F      0.000     0.000     0.000         1

    accuracy                          0.964      2595
   macro avg      0.563     0.686     0.588      2595
weighted avg      0.982     0.964

In [2]:
from sklearn.metrics import confusion_matrix

def diagnose(rec):
    record = wfdb.rdrecord(f"{DATA_DIR}/{rec}")
    ann = wfdb.rdann(f"{DATA_DIR}/{rec}", "atr")
    ch = record.sig_name.index("MLII")
    res = analyze(record.p_signal[:, ch], record.fs, bundle)

    keep = [i for i, s in enumerate(ann.symbol)
            if s in AAMI_MAP and AAMI_MAP[s] in CLASSES]
    a_samples = ann.sample[keep]
    a_labels = np.array([CLASSES.index(AAMI_MAP[ann.symbol[i]]) for i in keep])

    y_true, y_pred, offs = [], [], []
    for p, c in zip(res["peaks"], res["pred"]):
        j = np.argmin(np.abs(a_samples - p))
        off = a_samples[j] - p
        if abs(off) <= 36:
            y_true.append(a_labels[j])
            y_pred.append(c)
            offs.append(off)

    print(f"===== record {rec} | کانال‌ها: {record.sig_name} | انتخاب‌شده: {record.sig_name[ch]}")
    print("فاصله‌ی R-peak ما تا برچسب (نمونه): median =", int(np.median(offs)),
          "| بازه‌ی ۹۰٪:", int(np.percentile(offs, 5)), "تا", int(np.percentile(offs, 95)))
    print("ماتریس درهم‌ریختگی (ردیف = واقعی، ستون = پیش‌بینی، ترتیب N S V F):")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3]))

for rec in ["100", "200", "233"]:
    diagnose(rec)

===== record 100 | کانال‌ها: ['MLII', 'V5'] | انتخاب‌شده: MLII
فاصله‌ی R-peak ما تا برچسب (نمونه): median = 0 | بازه‌ی ۹۰٪: 0 تا 1
ماتریس درهم‌ریختگی (ردیف = واقعی، ستون = پیش‌بینی، ترتیب N S V F):
[[2237    0    0    0]
 [   0   33    0    0]
 [   0    0    1    0]
 [   0    0    0    0]]
===== record 200 | کانال‌ها: ['MLII', 'V1'] | انتخاب‌شده: MLII
فاصله‌ی R-peak ما تا برچسب (نمونه): median = 0 | بازه‌ی ۹۰٪: -1 تا 1
ماتریس درهم‌ریختگی (ردیف = واقعی، ستون = پیش‌بینی، ترتیب N S V F):
[[1712   30    0    0]
 [   5   25    0    0]
 [  17   39  764    2]
 [   1    0    0    0]]
===== record 233 | کانال‌ها: ['MLII', 'V1'] | انتخاب‌شده: MLII
فاصله‌ی R-peak ما تا برچسب (نمونه): median = 0 | بازه‌ی ۹۰٪: -1 تا 14
ماتریس درهم‌ریختگی (ردیف = واقعی، ستون = پیش‌بینی، ترتیب N S V F):
[[ 404    2    1 1820]
 [   1    4    1    1]
 [  30    1  779    7]
 [   5    0    2    4]]


In [3]:
import importlib
import src.pipeline
importlib.reload(src.pipeline)
from src.pipeline import analyze, DECISION_WEIGHTS

from src.features import build_features
from sklearn.metrics import classification_report, f1_score

# 1) نتیجه روی همه‌ی بیمارهای تست
d = np.load("../data/processed/beats.npz")
m = d["y_test"] != 4
F_test = build_features(d["X_test"][m], d["rr_test"][m], d["g_test"][m])
y_test = d["y_test"][m]

model = bundle["model"]
proba = np.zeros((len(y_test), 4))
proba[:, model.classes_] = model.predict_proba(F_test)
pred = (proba * DECISION_WEIGHTS).argmax(1)

print(classification_report(y_test, pred, labels=[0, 1, 2, 3],
                            target_names=CLASSES, digits=3, zero_division=0))
print("macro F1 روی N,S,V:", round(f1_score(y_test, pred, labels=[0, 1, 2], average="macro"), 3))

# 2) سه رکورد قبلی
for rec in ["100", "200", "233"]:
    diagnose(rec)

              precision    recall  f1-score   support

           N      0.956     0.971     0.963     44218
           S      0.181     0.161     0.170      1836
           V      0.946     0.918     0.932      3219
           F      0.000     0.000     0.000       388

    accuracy                          0.930     49661
   macro avg      0.521     0.512     0.516     49661
weighted avg      0.919     0.930     0.924     49661

macro F1 روی N,S,V: 0.689
===== record 100 | کانال‌ها: ['MLII', 'V5'] | انتخاب‌شده: MLII
فاصله‌ی R-peak ما تا برچسب (نمونه): median = 0 | بازه‌ی ۹۰٪: 0 تا 1
ماتریس درهم‌ریختگی (ردیف = واقعی، ستون = پیش‌بینی، ترتیب N S V F):
[[2237    0    0    0]
 [   0   33    0    0]
 [   0    0    1    0]
 [   0    0    0    0]]
===== record 200 | کانال‌ها: ['MLII', 'V1'] | انتخاب‌شده: MLII
فاصله‌ی R-peak ما تا برچسب (نمونه): median = 0 | بازه‌ی ۹۰٪: -1 تا 1
ماتریس درهم‌ریختگی (ردیف = واقعی، ستون = پیش‌بینی، ترتیب N S V F):
[[1712   30    0    0]
 [   5   25    0    0]
 [ 